# Notebook 1: LLM-Based Prompt Expansion (Deterministic)
Generate expanded prompts for all base prompts and emotions using Llama 3.2 1B Instruct (4-bit).

In [ ]:
# Cell 0: Install Dependencies
!pip install -q transformers accelerate bitsandbytes torch pandas tqdm

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import pandas as pd
from tqdm import tqdm
from google.colab import files
import os

print("✅ Dependencies installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.5 MB/s eta 0:00:00
✅ Dependencies installed


In [ ]:
# Cell 1: Hugging Face Login (required for Llama models)
from huggingface_hub import notebook_login
notebook_login()  # Paste your HF token with read access

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
# Cell 2: Configuration
BASE_PROMPTS = [
    "a person sitting alone in a room",
    "a beach at sunset",
    "a foggy mountain landscape",
    "a child playing in a park",
    "an office desk with a computer",
    "a street in a quiet neighborhood",
    "a cat sleeping on a couch",
    "a cup of coffee on a wooden table",
    "a library with many books",
    "a garden with flowers"
    'a child in a room'
]

EMOTIONS = ["joy", "calm", "sadness", "anxiety"]

In [ ]:
# Cell 3: Load BeautifulPrompt Model (Fixed 8-bit loading)
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_path = "alibaba-pai/pai-bloom-1b1-text2prompt-sd-v2"

# Proper 8-bit configuration
quant_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0,
)

tokenizer = AutoTokenizer.from_pretrained(model_path)

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    quantization_config=quant_config,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True
).eval()

print(f"✅ Successfully loaded BeautifulPrompt: {model_path}")

config.json:   0%|          | 0.00/806 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/96.0 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.13G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

✅ Successfully loaded BeautifulPrompt: alibaba-pai/pai-bloom-1b1-text2prompt-sd-v2


In [ ]:
# Cell 4: BeautifulPrompt Generation Function (Updated)
def generate_expansion(base_prompt: str, emotion: str, max_new_tokens=180):
    template = (
        "Converts a simple image description into a prompt. "
        "Prompts are formatted as multiple related tags separated by commas. "
        "Add appropriate words to make the images more aesthetically pleasing "
        "while keeping correlation with the input.\n"
        f"### Input: {base_prompt}, emotional atmosphere of {emotion}\n"
        "### Output:"
    )

    inputs = tokenizer(template, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=0.7,
            top_p=0.95,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.1
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    if "### Output:" in response:
        expanded = response.split("### Output:")[-1].strip()
    else:
        expanded = response.strip()

    expanded = expanded.split("\n")[0].strip().strip('"').strip("'")

    # Fallback
    if len(expanded) < 30:
        expanded = f"{base_prompt}, {emotion} mood, beautiful lighting, detailed"

    return expanded

In [ ]:
# Cell 5: Generate All Expansions with BeautifulPrompt
expansions = []

print("Generating prompts with BeautifulPrompt...")

for base_prompt in tqdm(BASE_PROMPTS):
    for emotion in EMOTIONS:
        expanded = generate_expansion(base_prompt, emotion)
        expansions.append({
            "base_prompt": base_prompt,
            "emotion": emotion,
            "expanded_prompt": expanded
        })

df_llm = pd.DataFrame(expansions)
df_llm.to_csv("llm_expanded_prompts.csv", index=False)

print("\n✅ BeautifulPrompt expansions saved to llm_expanded_prompts.csv")
display(df_llm.head(16))

files.download("llm_expanded_prompts.csv")

Generating prompts with BeautifulPrompt...


100%|██████████| 10/10 [03:54<00:00, 23.45s/it]


✅ BeautifulPrompt expansions saved to llm_expanded_prompts.csv


,base_prompt,emotion,expanded_prompt
0,a person sitting alone in a room,joy,"(absurdres, highres, ultra detailed), 1person,..."
1,a person sitting alone in a room,calm,"(extremely detailed CG unity 8k wallpaper), a ..."
2,a person sitting alone in a room,sadness,"(absurdres, highres, ultra detailed), 1person,..."
3,a person sitting alone in a room,anxiety,"best quality, masterpiece, ultra high res, (ph..."
4,a beach at sunset,joy,"beach, sunset, sun, smile, happiness, ocean, c..."
5,a beach at sunset,calm,"beach, sunset, calm, beautiful, aesthetic, ext..."
6,a beach at sunset,sadness,"beach, sunset, sadness, emotional, cinematic, ..."
7,a beach at sunset,anxiety,"beach, sunset, calm, beautiful, peaceful, sun ..."
8,a foggy mountain landscape,joy,A stunningly beautiful foggy mountain landscap...
9,a foggy mountain landscape,calm,"A foggy mountain landscape, with a sense of ca..."


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Cell 6: Save and Preview
df_llm = pd.DataFrame(expansions)
df_llm.to_csv("llm_expanded_prompts.csv", index=False)

print("\n✅ LLM expansions saved to llm_expanded_prompts.csv")
print("\nPreview:")
display(df_llm.head(12))

# Download
files.download("llm_expanded_prompts.csv")


✅ LLM expansions saved to llm_expanded_prompts.csv

Preview:


,base_prompt,emotion,expanded_prompt
0,a person sitting alone in a room,joy,"(absurdres, highres, ultra detailed), 1person,..."
1,a person sitting alone in a room,calm,"(extremely detailed CG unity 8k wallpaper), a ..."
2,a person sitting alone in a room,sadness,"(absurdres, highres, ultra detailed), 1person,..."
3,a person sitting alone in a room,anxiety,"best quality, masterpiece, ultra high res, (ph..."
4,a beach at sunset,joy,"beach, sunset, sun, smile, happiness, ocean, c..."
5,a beach at sunset,calm,"beach, sunset, calm, beautiful, aesthetic, ext..."
6,a beach at sunset,sadness,"beach, sunset, sadness, emotional, cinematic, ..."
7,a beach at sunset,anxiety,"beach, sunset, calm, beautiful, peaceful, sun ..."
8,a foggy mountain landscape,joy,A stunningly beautiful foggy mountain landscap...
9,a foggy mountain landscape,calm,"A foggy mountain landscape, with a sense of ca..."


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>